# Extended evaluation + L+S baseline (odls_eval_plus)

Runs `odls_eval_plus/` on the models you already trained -- **no training, nothing in
`odls_dynamic/` or in your run folders is changed**. Everything new goes to
`MyDrive/odls_dynamic/eval_plus/`. See `odls_eval_plus/README.md` for what every number means.

What it does (about 30-60 min on a T4):
1. tunes L+S (the classical low-rank + sparse method) on the **validation** slices, for AF 4 and AF 8
2. scores zero-filled, L+S, frame-by-frame ODLS and dynamic ODLS on the **test** slices with
   RLNE, PSNR, global SSIM (paper), windowed SSIM (standard), time, parameters, GPU memory
3. builds the final tables with Wilcoxon significance tests
4. draws the side-by-side figures

**How to run:** use the account that owns `odls_dynamic` -> Runtime -> Change runtime type ->
**T4 GPU** -> **Runtime -> Run all**. Safe to re-run: finished steps are skipped
(delete a result folder in `eval_plus/` to redo that step).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/odls_dynamic"
DRIVE_DATA_DIR = f"{DRIVE_ROOT}/ocmr_preprocessed"
LOCAL_DATA_DIR = "/content/ocmr_preprocessed"        # fast local copy, wiped with the runtime
EVAL_ROOT = f"{DRIVE_ROOT}/eval_plus"                # NEW folder; training runs are not touched

REPO_URL = "https://github.com/Shambhawi419/1D_MRI.git"
REPO_DIR = "/content/1D_MRI"
CODE_DIR = f"{REPO_DIR}/odls_eval_plus"

AFS = [4, 8]
# The trained models to score (their best.pt), per AF: frame-by-frame first, then dynamic.
MODELS = {
    4: [f"{DRIVE_ROOT}/runs/dyn_af4_kt1_yt_v2/best.pt", f"{DRIVE_ROOT}/runs/dyn_af4_kt3_yt_v2/best.pt"],
    8: [f"{DRIVE_ROOT}/runs/dyn_af8_kt1_yt_v2/best.pt", f"{DRIVE_ROOT}/runs/dyn_af8_kt3_yt_v2/best.pt"],
}

In [ ]:
import os, subprocess, sys
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

from google.colab import drive


def ensure_drive():
    # If Drive is not mounted, /content/drive/... is an empty LOCAL folder and results would vanish.
    if not os.path.ismount('/content/drive'):
        print("Google Drive is not connected -- reconnecting (accept the pop-up if shown) ...")
        drive.mount('/content/drive', force_remount=True)
    if not os.path.isdir(DRIVE_DATA_DIR):
        raise RuntimeError(f"{DRIVE_DATA_DIR} not found -- is this the account that owns odls_dynamic?")


def flush_drive():
    # Waits until everything written to Drive is uploaded. Don't interrupt it.
    print("flushing Google Drive (waiting for uploads to finish -- don't stop this cell) ...")
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
    print("Drive flushed -- everything is saved.")


!nvidia-smi --query-gpu=name,memory.total --format=csv

## Optional: self-test (under 1 min, no data needed)
Checks windowed SSIM, the L+S algorithm, that the masks/slices match `odls_dynamic`'s
evaluation exactly, and the whole pipeline on synthetic data.

In [ ]:
!cd {CODE_DIR} && python -u selftest_plus.py

## 0. Check the data and the trained models, then copy the data to local disk

In [ ]:
import csv
ensure_drive()
rows = list(csv.DictReader(open(f"{DRIVE_DATA_DIR}/manifest.csv")))
print(f"manifest: {len(rows)} slices")
assert len(rows) == 128, "preprocessed data incomplete"
for af in AFS:
    for ckpt in MODELS[af]:
        print(("found   " if os.path.exists(ckpt) else "MISSING ") + ckpt)
        assert os.path.exists(ckpt), f"train this model first: {ckpt}"

In [ ]:
import shutil, time


def remount():
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)


src_dir, dst_dir = f"{DRIVE_DATA_DIR}/slices", f"{LOCAL_DATA_DIR}/slices"
os.makedirs(dst_dir, exist_ok=True)
for attempt in range(1, 6):
    try:
        names = sorted(n for n in os.listdir(src_dir) if n.endswith(".npy"))
        copied = 0
        for name in names:
            src, dst = f"{src_dir}/{name}", f"{dst_dir}/{name}"
            if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
                continue
            shutil.copy(src, dst + ".tmp")
            os.replace(dst + ".tmp", dst)
            copied += 1
        for name in ("manifest.csv", "split.csv"):
            shutil.copy(f"{DRIVE_DATA_DIR}/{name}", f"{LOCAL_DATA_DIR}/{name}")
        print(f"done: copied {copied} new files, {len(names)} slices available locally")
        break
    except OSError as e:
        print(f"Drive connection dropped ({e}); reconnecting, attempt {attempt}/5 ...")
        time.sleep(5)
        remount()
else:
    raise RuntimeError("Drive kept disconnecting -- restart the runtime and run this cell again")

## 1. Tune L+S on the validation slices (~3-10 min per AF)
Tries a grid of (lam_L, lam_S) on the 13 validation slices and keeps the pair with the lowest
RLNE. If the best pair is on the edge of the grid, the grid is widened automatically (up to 3
times) until it is inside. Results tuned by an older version of this notebook (or still on the
grid edge) are re-tuned, and their test results for L+S are redone in step 2.

In [ ]:
import json, shutil
ensure_drive()
for af in AFS:
    params = f"{EVAL_ROOT}/lps_params_af{af}.json"
    if os.path.exists(params):
        old = json.load(open(params))
        if old.get("version", 1) >= 2 and not old.get("best_on_grid_edge"):
            print(f"AF {af}: L+S already tuned (lam_L={old['best']['lam_l']}, lam_S={old['best']['lam_s']}) -- skipping")
            continue
        print(f"AF {af}: re-tuning L+S with the auto-widening grid (old result was on the grid edge / older version)")
        os.remove(params)
        shutil.rmtree(f"{EVAL_ROOT}/test_af{af}/lps", ignore_errors=True)   # its test score must be redone too
    get_ipython().system(f"cd {CODE_DIR} && python -u tune_lps.py --data-dir {LOCAL_DATA_DIR} "
                         f"--out-root {EVAL_ROOT} --af {af}")
flush_drive()

## 2. Score every method on the test slices (~5-10 min)
zero-filled, L+S, frame-by-frame ODLS and dynamic ODLS, at AF 4 and AF 8 -- same slices and
same masks for all. Each result goes to `eval_plus/test_af<AF>/<method>/results_plus.json`.

In [ ]:
ensure_drive()
for af in AFS:
    base = (f"cd {CODE_DIR} && python -u evaluate_plus.py --data-dir {LOCAL_DATA_DIR} "
            f"--out-root {EVAL_ROOT} --split test")
    jobs = [("zero_filled", f"--method zero_filled --af {af}"),
            ("lps", f"--method lps --af {af} --lps-params {EVAL_ROOT}/lps_params_af{af}.json")]
    jobs += [(os.path.basename(os.path.dirname(c)), f"--method model --checkpoint {c}") for c in MODELS[af]]
    for name, extra in jobs:
        if os.path.exists(f"{EVAL_ROOT}/test_af{af}/{name}/results_plus.json"):
            print(f"AF {af} {name}: already scored -- skipping")
            continue
        get_ipython().system(f"{base} {extra}")
flush_drive()

## 3. Final tables + significance tests

In [ ]:
from IPython.display import Markdown, display
get_ipython().system(f"cd {CODE_DIR} && python -u compare_plus.py --out-root {EVAL_ROOT} > /dev/null")
for af in AFS:
    path = f"{EVAL_ROOT}/summary_test_af{af}.md"
    if os.path.exists(path):
        text = open(path, encoding="utf-8").read()
        display(Markdown(text))            # formatted table
        print(text)                        # same table as plain text, easy to copy

## 4. Side-by-side figures (same slice, same mask, every method)
The y-t profile (bottom row) runs through the column with the most heart motion. Note the thin
vertical line at the image centre in the ODLS reconstructions: a known artifact of the k-space
low-rank module (largest at AF 8), not a plotting error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for af in AFS:
    folders = {"zero-filled": "zero_filled", "L+S": "lps"}
    for ckpt in MODELS[af]:
        run = os.path.basename(os.path.dirname(ckpt))
        folders["dynamic ODLS" if "_kt3_" in run else "frame-by-frame ODLS"] = run
    dirs = {label: f"{EVAL_ROOT}/test_af{af}/{sub}" for label, sub in folders.items()}
    if not all(os.path.isdir(d) for d in dirs.values()):
        print(f"AF {af}: run step 2 first")
        continue
    common = sorted(set.intersection(*[set(n for n in os.listdir(d) if n.endswith(".npz")) for d in dirs.values()]))
    name = common[0]
    data = {label: np.load(f"{d}/{name}") for label, d in dirs.items()}
    ref = data["zero-filled"]["ref"].astype(float)
    t, vmax = ref.shape[0] // 2, ref.max()
    # y-t profile through the column with the most motion (largest change over time), avoiding
    # the image's centre column, where the k-space networks leave a thin line artifact
    motion = ref.std(axis=0).mean(axis=0)                    # per image column
    centre = ref.shape[2] // 2
    motion[max(0, centre - 3):centre + 4] = 0
    col = int(np.argmax(motion))
    panels = [("fully sampled", ref)] + [(label, data[label]["est"].astype(float)) for label in
                                         ["zero-filled", "L+S", "frame-by-frame ODLS", "dynamic ODLS"]]
    fig, ax = plt.subplots(3, 5, figsize=(20, 12))
    for j, (title, img) in enumerate(panels):
        ax[0, j].imshow(img[t], cmap="gray", vmin=0, vmax=vmax); ax[0, j].set_title(f"{title}, frame {t}")
        ax[1, j].imshow(np.abs(img[t] - ref[t]) * 5, cmap="gray", vmin=0, vmax=vmax); ax[1, j].set_title("|error| x5")
        ax[2, j].imshow(img[:, :, col].T, cmap="gray", vmin=0, vmax=vmax, aspect="auto"); ax[2, j].set_title(f"y-t profile (column {col})")
    for a in ax.ravel():
        a.axis("off")
    plt.suptitle(f"{name.replace('.npz', '')}, AF {af}", fontsize=14); plt.tight_layout(); plt.show()

## Notes
- Re-running is safe: finished steps are skipped. To redo one, delete its folder in
  `MyDrive/odls_dynamic/eval_plus/` (e.g. `test_af8/lps/`) and run again.
- The tables are also saved as `eval_plus/summary_test_af4.md` / `summary_test_af8.md`.
- `odls_dynamic/` and the run folders under `odls_dynamic/runs/` are only read, never written.